I used this file to vectorize image and text data from extracted_data and store this under the storage folder. I used LlamaIndex to implement chunking and the storage of these chunks with important identifying metadata information for each chunk. 

In [34]:
import pandas as pd
from dotenv import load_dotenv
import datauri
import os
import re
import json
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from typing import Optional
from llama_index.core.schema import TextNode
from llama_index.core.node_parser import MarkdownNodeParser
from typing import List
import qdrant_client
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import ImageNode

In [38]:
#Function to create image nodes
json_file_list = []
for i in range(1,33):
    json_file_list.append(f"extracted_data/document_{i}/metadata_{i}.json")
def get_img_nodes(json_file_list):
    nodes=[]
    for file in json_file_list:
        with open(file,"r") as f:
            data = json.load(f)
        for d in data:
            node = ImageNode(
                text=d["img_description"],
                image_path=d["img_path"],
                metadata={
                    "doc_path":d["doc_path"],
                    "ref_id":d["ref_id"],
                    "ref_url":d["ref_url"],
                    "img_id":d["img_id"],
                    "img_uid":d["img_uid"],
                    "context":d["context"]
                },
                excluded_embed_metadata_keys = ["doc_path","ref_id","ref_url","img_id","img_uid",
                                                "context","img_path"],
                excluded_llm_metadata_keys = ["img_path","img_id","img_uid","doc_path"]
            )
            nodes.append(node)
    return nodes

#Function to create text nodes
md_file_list = []
for i in range(1,33):
    md_file_list.append(f"extracted_data/document_{i}/doc_{i}.md")
parser = MarkdownNodeParser()
def get_md_nodes(md_file_list):
    nodes=[]
    for file in md_file_list:
        document = SimpleDirectoryReader(input_files=[file]).load_data()
        node = parser.get_nodes_from_documents(document)
        json_path = f"extracted_data/document_{md_file_list.index(file)+1}/doc_data_{md_file_list.index(file)+1}.json"
        with open(json_path,"r") as f:
            json_data = json.load(f)
            ref_id = json_data.get("ref_id")
            ref_url = json_data.get("ref_url")
        for n in node:
            n.metadata["ref_id"] = ref_id
            n.metadata["ref_url"] = ref_url
        nodes.extend(node)
    return nodes

In [43]:
image_nodes = get_img_nodes(json_file_list)
text_nodes = get_md_nodes(md_file_list)
splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
temp_docs = [
    Document(text=n.get_content(), metadata=n.metadata) 
    for n in text_nodes
]
condensed_text_nodes = splitter.get_nodes_from_documents(temp_docs)
nodes = image_nodes + condensed_text_nodes
index = VectorStoreIndex(nodes)  #Creating the vector store.
index.storage_context.persist(persist_dir="./storage")  #Saves the vector store so we don't have to repeat this process. 

2026-01-31 19:10:41,798 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:42,534 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:43,090 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:43,848 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:44,521 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:44,977 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:45,894 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:46,817 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:47,333 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-31 19:10:48,250 - INFO - HTTP